# 11 - TTA + Attention + Transformer: El Asalto Final al 60%

## Estrategias

1. **TTA (Test-Time Augmentation)**: +1-3% GRATIS sin reentrenar
2. **GRU con Attention**: Foco en frames importantes
3. **Transformer Encoder**: Arquitectura moderna
4. **Ensemble Final**: Combinar todos los modelos

## Estado actual
- Ensemble LSTM+GRU+GRU_Enriched: ~55% Val, ~50% Test (aproximado)

In [1]:
# ==========================================
# 1. IMPORTS
# ==========================================
import numpy as np
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
# ==========================================
# 2. RUTAS Y CONFIGURACION
# ==========================================
BASE = Path("/home/fransando/Dev/tfg-sign-language-recognition")
EXPERIMENTS = BASE / "05_experiments"
DATASETS = BASE / "06_datasets/wlasl/landmarks"

# Modelos existentes
LSTM_PATH = EXPERIMENTS / "lstm_landmarks/best_model.pt"
GRU_PATH = EXPERIMENTS / "lstm_landmarks_wlasl30_gru_suave/best_model.pt"
GRU_ENRICHED_PATH = EXPERIMENTS / "ensemble_final/best_model_enriched.pt"

# Datasets
WLASL30_ROOT = DATASETS / "wlasl30"
FILTERED_ROOT = DATASETS / "wlasl100_filtered30"

# Output
OUTPUT_DIR = EXPERIMENTS / "final_assault"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verificar
print("Modelos disponibles:")
print(f"  LSTM: {LSTM_PATH.exists()}")
print(f"  GRU: {GRU_PATH.exists()}")
print(f"  GRU Enriched: {GRU_ENRICHED_PATH.exists()}")

Modelos disponibles:
  LSTM: True
  GRU: True
  GRU Enriched: True


In [3]:
# ==========================================
# 3. CARGAR METADATA
# ==========================================
with open(WLASL30_ROOT / 'metadata.json') as f:
    meta = json.load(f)

gloss_to_idx = meta['gloss_to_idx']
idx_to_gloss = {int(k): v for k, v in meta['idx_to_gloss'].items()}
NUM_CLASSES = len(gloss_to_idx)

print(f"Clases: {NUM_CLASSES}")

Clases: 30


In [4]:
# ==========================================
# 4. DATASET CON SOPORTE PARA TTA
# ==========================================

class PoseDatasetTTA(Dataset):
    """
    Dataset con soporte para Test-Time Augmentation.
    Puede devolver multiples versiones augmentadas del mismo sample.
    """
    
    def __init__(self, root_dir: Path, split: str, gloss_to_idx: Dict,
                 max_frames: int = 64, augmentation=None, tta_transforms=None):
        self.root_dir = root_dir
        self.max_frames = max_frames
        self.augmentation = augmentation
        self.tta_transforms = tta_transforms  # Lista de transforms para TTA
        self.gloss_to_idx = gloss_to_idx
        
        self.samples = []
        split_dir = root_dir / split
        
        for gloss_dir in split_dir.iterdir():
            if not gloss_dir.is_dir():
                continue
            gloss = gloss_dir.name
            if gloss not in gloss_to_idx:
                continue
            label = gloss_to_idx[gloss]
            
            for npy_file in gloss_dir.glob('*.npy'):
                self.samples.append((npy_file, label, gloss))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        npy_path, label, gloss = self.samples[idx]
        landmarks = np.load(npy_path)
        landmarks = self._pad_or_truncate(landmarks)
        
        if self.augmentation:
            landmarks = self.augmentation(landmarks)
        
        landmarks = landmarks.reshape(self.max_frames, -1)
        return torch.tensor(landmarks, dtype=torch.float32), label
    
    def get_tta_batch(self, idx, n_augments=5):
        """Devuelve N versiones augmentadas del mismo sample para TTA."""
        npy_path, label, gloss = self.samples[idx]
        landmarks_orig = np.load(npy_path)
        landmarks_orig = self._pad_or_truncate(landmarks_orig)
        
        batch = []
        
        # Version original
        lm = landmarks_orig.reshape(self.max_frames, -1)
        batch.append(torch.tensor(lm, dtype=torch.float32))
        
        # Versiones augmentadas
        if self.tta_transforms:
            for transform in self.tta_transforms[:n_augments-1]:
                lm_aug = transform(landmarks_orig.copy())
                lm_aug = lm_aug.reshape(self.max_frames, -1)
                batch.append(torch.tensor(lm_aug, dtype=torch.float32))
        
        return torch.stack(batch), label
    
    def _pad_or_truncate(self, landmarks):
        T = landmarks.shape[0]
        if T > self.max_frames:
            indices = np.linspace(0, T-1, self.max_frames, dtype=int)
            landmarks = landmarks[indices]
        elif T < self.max_frames:
            pad_size = self.max_frames - T
            padding = np.zeros((pad_size, 21, 3))
            landmarks = np.concatenate([landmarks, padding], axis=0)
        return landmarks

print("PoseDatasetTTA definido.")

PoseDatasetTTA definido.


In [5]:
# ==========================================
# 5. TTA TRANSFORMS
# ==========================================

class TTATransform:
    """Transformaciones suaves para TTA."""
    
    @staticmethod
    def rotate(landmarks, angle_deg):
        landmarks = landmarks.copy()
        angle = np.radians(angle_deg)
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        x, y = landmarks[:, :, 0].copy(), landmarks[:, :, 1].copy()
        landmarks[:, :, 0] = x * cos_a - y * sin_a
        landmarks[:, :, 1] = x * sin_a + y * cos_a
        return landmarks
    
    @staticmethod
    def scale(landmarks, factor):
        return landmarks * factor
    
    @staticmethod
    def flip_horizontal(landmarks):
        landmarks = landmarks.copy()
        landmarks[:, :, 0] = -landmarks[:, :, 0]
        return landmarks
    
    @staticmethod
    def add_noise(landmarks, std=0.005):
        return landmarks + np.random.normal(0, std, landmarks.shape)
    
    @staticmethod
    def translate(landmarks, tx, ty):
        landmarks = landmarks.copy()
        landmarks[:, :, 0] += tx
        landmarks[:, :, 1] += ty
        return landmarks


# Crear lista de transforms para TTA
TTA_TRANSFORMS = [
    lambda x: x,  # Original
    lambda x: TTATransform.rotate(x, 5),
    lambda x: TTATransform.rotate(x, -5),
    lambda x: TTATransform.scale(x, 0.95),
    lambda x: TTATransform.scale(x, 1.05),
    lambda x: TTATransform.flip_horizontal(x),
    lambda x: TTATransform.add_noise(x, 0.005),
    lambda x: TTATransform.translate(x, 0.02, 0),
    lambda x: TTATransform.translate(x, -0.02, 0),
    lambda x: TTATransform.translate(x, 0, 0.02),
]

print(f"TTA Transforms: {len(TTA_TRANSFORMS)} variaciones")

TTA Transforms: 10 variaciones


In [6]:
# ==========================================
# 6. MODELOS BASE (LSTM y GRU)
# ==========================================

class PoseClassifier(nn.Module):
    """Modelo base LSTM/GRU."""
    
    def __init__(self, config: Dict):
        super().__init__()
        
        input_size = config['input_size']
        hidden_size = config['hidden_size']
        num_layers = config['num_layers']
        num_classes = config['num_classes']
        dropout = config['dropout']
        bidirectional = config['bidirectional']
        model_type = config.get('model_type', 'LSTM')
        
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        RNN_CLASS = nn.LSTM if model_type == 'LSTM' else nn.GRU
        self.rnn = RNN_CLASS(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        self.model_type = model_type
        
        rnn_output_size = hidden_size * self.num_directions
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(rnn_output_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_projection(x)
        
        if self.model_type == 'LSTM':
            rnn_out, (h_n, c_n) = self.rnn(x)
        else:
            rnn_out, h_n = self.rnn(x)
        
        if self.bidirectional:
            h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h_final = h_n[-1]
        
        return self.classifier(h_final)

print("PoseClassifier definido.")

PoseClassifier definido.


In [7]:
# ==========================================
# 7. GRU CON ATTENTION
# ==========================================

class TemporalAttention(nn.Module):
    """Mecanismo de atencion temporal para secuencias."""
    
    def __init__(self, hidden_size: int):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1, bias=False)
        )
    
    def forward(self, rnn_output: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            rnn_output: (batch, seq_len, hidden_size)
        Returns:
            context: (batch, hidden_size) - weighted sum
            weights: (batch, seq_len) - attention weights
        """
        # Calcular scores de atencion
        scores = self.attention(rnn_output)  # (batch, seq_len, 1)
        scores = scores.squeeze(-1)  # (batch, seq_len)
        
        # Softmax para obtener pesos
        weights = F.softmax(scores, dim=1)  # (batch, seq_len)
        
        # Weighted sum
        context = torch.bmm(weights.unsqueeze(1), rnn_output)  # (batch, 1, hidden)
        context = context.squeeze(1)  # (batch, hidden)
        
        return context, weights


class PoseClassifierAttention(nn.Module):
    """
    GRU con Attention: El modelo aprende a enfocarse en los frames importantes.
    """
    
    def __init__(self, config: Dict):
        super().__init__()
        
        input_size = config['input_size']
        hidden_size = config['hidden_size']
        num_layers = config['num_layers']
        num_classes = config['num_classes']
        dropout = config['dropout']
        bidirectional = config.get('bidirectional', True)
        
        self.num_directions = 2 if bidirectional else 1
        rnn_hidden = hidden_size * self.num_directions
        
        # Proyeccion de entrada
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # GRU
        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        # Attention
        self.attention = TemporalAttention(rnn_hidden)
        
        # Clasificador
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(rnn_hidden, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )
    
    def forward(self, x: torch.Tensor, return_attention: bool = False):
        x = self.input_projection(x)
        rnn_out, _ = self.gru(x)  # (batch, seq_len, hidden*2)
        
        context, attn_weights = self.attention(rnn_out)
        logits = self.classifier(context)
        
        if return_attention:
            return logits, attn_weights
        return logits


print("PoseClassifierAttention definido.")

PoseClassifierAttention definido.


In [8]:
# ==========================================
# 8. TRANSFORMER ENCODER
# ==========================================

class PositionalEncoding(nn.Module):
    """Positional encoding para Transformer."""
    
    def __init__(self, d_model: int, max_len: int = 65, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class PoseTransformer(nn.Module):
    """
    Transformer Encoder para clasificacion de poses.
    Arquitectura moderna, potente pero requiere mas datos.
    """
    
    def __init__(self, config: Dict):
        super().__init__()
        
        input_size = config['input_size']
        d_model = config.get('d_model', 256)
        nhead = config.get('nhead', 8)
        num_layers = config.get('num_transformer_layers', 4)
        num_classes = config['num_classes']
        dropout = config['dropout']
        max_len = config.get('max_frames', 64)
        
        # Proyeccion de entrada
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, d_model),
            nn.LayerNorm(d_model)
        )
        
        # Positional encoding (max_len + 1 para el CLS token)
        self.pos_encoder = PositionalEncoding(d_model, max_len + 1, dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # CLS token (learnable)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Clasificador
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.size(0)
        
        # Proyectar entrada
        x = self.input_projection(x)  # (batch, seq_len, d_model)
        
        # Añadir CLS token al principio
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (batch, seq_len+1, d_model)
        
        # Positional encoding
        x = self.pos_encoder(x)
        
        # Transformer
        x = self.transformer(x)
        
        # Usar CLS token para clasificacion
        cls_output = x[:, 0]  # (batch, d_model)
        
        return self.classifier(cls_output)


print("PoseTransformer definido (corregido para CLS token).")

PoseTransformer definido (corregido para CLS token).


In [9]:
# ==========================================
# 9. CARGAR MODELOS EXISTENTES
# ==========================================

models_loaded = {}

# LSTM
if LSTM_PATH.exists():
    ckpt = torch.load(LSTM_PATH, map_location=device)
    config = ckpt['config'].copy()
    config['model_type'] = 'LSTM'
    model = PoseClassifier(config).to(device)
    # Rename keys from 'lstm.*' to 'rnn.*' if needed
    state_dict = ckpt['model_state_dict']
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('lstm.'):
            new_state_dict['rnn.' + k[5:]] = v
        else:
            new_state_dict[k] = v
    model.load_state_dict(new_state_dict)
    model.eval()
    models_loaded['LSTM'] = model
    print(f"LSTM cargado")

# GRU
if GRU_PATH.exists():
    ckpt = torch.load(GRU_PATH, map_location=device)
    config = ckpt['config'].copy()
    model = PoseClassifier(config).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    models_loaded['GRU'] = model
    print(f"GRU cargado")

# GRU Enriched
if GRU_ENRICHED_PATH.exists():
    ckpt = torch.load(GRU_ENRICHED_PATH, map_location=device)
    config = ckpt['config'].copy()
    model = PoseClassifier(config).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    models_loaded['GRU_Enriched'] = model
    print(f"GRU Enriched cargado")

print(f"\nModelos cargados: {list(models_loaded.keys())}")

LSTM cargado
GRU cargado
GRU Enriched cargado

Modelos cargados: ['LSTM', 'GRU', 'GRU_Enriched']


In [10]:
# ==========================================
# 10. CREAR DATASETS
# ==========================================

# Dataset normal
val_dataset = PoseDatasetTTA(WLASL30_ROOT, 'val', gloss_to_idx, tta_transforms=TTA_TRANSFORMS)
test_dataset = PoseDatasetTTA(WLASL30_ROOT, 'test', gloss_to_idx, tta_transforms=TTA_TRANSFORMS)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Val: {len(val_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

Val: 56 samples
Test: 34 samples


---
# PARTE 1: TEST-TIME AUGMENTATION (TTA)
---

In [11]:
# ==========================================
# 11. FUNCIONES DE EVALUACION
# ==========================================

@torch.no_grad()
def evaluate_standard(models: List[nn.Module], loader: DataLoader, 
                      weights: List[float] = None, top_k=(1, 5)):
    """Evaluacion estandar (sin TTA)."""
    if weights is None:
        weights = [1.0 / len(models)] * len(models)
    
    for m in models:
        m.eval()
    
    total = 0
    top_k_correct = {k: 0 for k in top_k}
    
    for batch, labels in loader:
        batch, labels = batch.to(device), labels.to(device)
        
        ensemble_logits = None
        for model, w in zip(models, weights):
            logits = model(batch)
            if ensemble_logits is None:
                ensemble_logits = w * logits
            else:
                ensemble_logits += w * logits
        
        total += labels.size(0)
        _, pred_topk = ensemble_logits.topk(max(top_k), 1, True, True)
        for k in top_k:
            correct_k = pred_topk[:, :k].eq(labels.view(-1, 1).expand_as(pred_topk[:, :k]))
            top_k_correct[k] += correct_k.any(dim=1).sum().item()
    
    return {f'top{k}_acc': 100 * top_k_correct[k] / total for k in top_k}


@torch.no_grad()
def evaluate_with_tta(models: List[nn.Module], dataset: PoseDatasetTTA,
                      weights: List[float] = None, n_augments: int = 5, top_k=(1, 5)):
    """
    Evaluacion con Test-Time Augmentation.
    Para cada sample, genera N versiones augmentadas y promedia predicciones.
    """
    if weights is None:
        weights = [1.0 / len(models)] * len(models)
    
    for m in models:
        m.eval()
    
    total = 0
    top_k_correct = {k: 0 for k in top_k}
    
    for idx in tqdm(range(len(dataset)), desc="TTA Evaluation"):
        # Obtener batch de versiones augmentadas
        tta_batch, label = dataset.get_tta_batch(idx, n_augments)
        tta_batch = tta_batch.to(device)  # (n_augments, seq_len, features)
        
        # Promediar predicciones de todos los modelos y augmentaciones
        ensemble_logits = None
        
        for model, w in zip(models, weights):
            # Predecir para todas las augmentaciones
            logits = model(tta_batch)  # (n_augments, num_classes)
            # Promediar sobre augmentaciones
            avg_logits = logits.mean(dim=0, keepdim=True)  # (1, num_classes)
            
            if ensemble_logits is None:
                ensemble_logits = w * avg_logits
            else:
                ensemble_logits += w * avg_logits
        
        total += 1
        _, pred_topk = ensemble_logits.topk(max(top_k), 1, True, True)
        label_tensor = torch.tensor([label], device=device)
        
        for k in top_k:
            correct_k = pred_topk[:, :k].eq(label_tensor.view(-1, 1).expand_as(pred_topk[:, :k]))
            top_k_correct[k] += correct_k.any(dim=1).sum().item()
    
    return {f'top{k}_acc': 100 * top_k_correct[k] / total for k in top_k}


print("Funciones de evaluacion definidas.")

Funciones de evaluacion definidas.


In [12]:
# ==========================================
# 12. COMPARAR: SIN TTA vs CON TTA
# ==========================================

print("=" * 60)
print("COMPARACION: SIN TTA vs CON TTA")
print("=" * 60)

# Modelos a usar
model_list = list(models_loaded.values())
model_names = list(models_loaded.keys())

print(f"\nUsando {len(model_list)} modelos: {model_names}")

# Sin TTA
print("\n--- Evaluacion SIN TTA ---")
val_standard = evaluate_standard(model_list, val_loader)
test_standard = evaluate_standard(model_list, test_loader)
print(f"Val:  Top-1: {val_standard['top1_acc']:.1f}%  Top-5: {val_standard['top5_acc']:.1f}%")
print(f"Test: Top-1: {test_standard['top1_acc']:.1f}%  Top-5: {test_standard['top5_acc']:.1f}%")

# Con TTA
print("\n--- Evaluacion CON TTA (5 augmentaciones) ---")
val_tta = evaluate_with_tta(model_list, val_dataset, n_augments=5)
test_tta = evaluate_with_tta(model_list, test_dataset, n_augments=5)
print(f"Val:  Top-1: {val_tta['top1_acc']:.1f}%  Top-5: {val_tta['top5_acc']:.1f}%")
print(f"Test: Top-1: {test_tta['top1_acc']:.1f}%  Top-5: {test_tta['top5_acc']:.1f}%")

# Mejora
print(f"\n--- MEJORA CON TTA ---")
print(f"Val:  {val_tta['top1_acc'] - val_standard['top1_acc']:+.1f}%")
print(f"Test: {test_tta['top1_acc'] - test_standard['top1_acc']:+.1f}%")

COMPARACION: SIN TTA vs CON TTA

Usando 3 modelos: ['LSTM', 'GRU', 'GRU_Enriched']

--- Evaluacion SIN TTA ---
Val:  Top-1: 50.0%  Top-5: 80.4%
Test: Top-1: 58.8%  Top-5: 79.4%

--- Evaluacion CON TTA (5 augmentaciones) ---


TTA Evaluation: 100%|██████████| 34/34 [00:00<00:00, 84.91it/s]

Val:  Top-1: 50.0%  Top-5: 80.4%
Test: Top-1: 58.8%  Top-5: 79.4%

--- MEJORA CON TTA ---
Val:  +0.0%
Test: +0.0%


In [13]:
# ==========================================
# 13. TTA CON MAS AUGMENTACIONES
# ==========================================

print("\n--- TTA con diferentes numeros de augmentaciones ---")

for n_aug in [3, 5, 7, 10]:
    test_tta_n = evaluate_with_tta(model_list, test_dataset, n_augments=n_aug)
    print(f"N={n_aug}: Test Top-1: {test_tta_n['top1_acc']:.1f}%  Top-5: {test_tta_n['top5_acc']:.1f}%")


--- TTA con diferentes numeros de augmentaciones ---


TTA Evaluation: 100%|██████████| 34/34 [00:00<00:00, 95.54it/s]


N=3: Test Top-1: 55.9%  Top-5: 79.4%


TTA Evaluation: 100%|██████████| 34/34 [00:00<00:00, 87.73it/s]


N=5: Test Top-1: 58.8%  Top-5: 79.4%


TTA Evaluation: 100%|██████████| 34/34 [00:00<00:00, 81.97it/s]


N=7: Test Top-1: 58.8%  Top-5: 79.4%


TTA Evaluation: 100%|██████████| 34/34 [00:00<00:00, 84.46it/s]

N=10: Test Top-1: 58.8%  Top-5: 79.4%


---
# PARTE 2: ENTRENAR GRU CON ATTENTION
---

In [14]:
# ==========================================
# 14. AUGMENTATION SUAVE PARA ENTRENAMIENTO
# ==========================================

class SkeletalAugmentation:
    def __init__(self, rotation=10, scale=(0.9, 1.1), translation=0.1, noise=0.01, p=0.5):
        self.rotation = rotation
        self.scale = scale
        self.translation = translation
        self.noise = noise
        self.p = p
    
    def __call__(self, landmarks):
        landmarks = landmarks.copy()
        
        if random.random() < self.p:
            angle = np.radians(random.uniform(-self.rotation, self.rotation))
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            x, y = landmarks[:, :, 0].copy(), landmarks[:, :, 1].copy()
            landmarks[:, :, 0] = x * cos_a - y * sin_a
            landmarks[:, :, 1] = x * sin_a + y * cos_a
        
        if random.random() < self.p:
            landmarks = landmarks * random.uniform(*self.scale)
        
        if random.random() < self.p:
            landmarks[:, :, 0] += random.uniform(-self.translation, self.translation)
            landmarks[:, :, 1] += random.uniform(-self.translation, self.translation)
        
        if random.random() < self.p:
            landmarks = landmarks + np.random.normal(0, self.noise, landmarks.shape)
        
        if random.random() < self.p * 0.5:
            landmarks[:, :, 0] = -landmarks[:, :, 0]
        
        return landmarks

In [15]:
# ==========================================
# 15. CONFIGURACION GRU+ATTENTION
# ==========================================

CONFIG_ATTENTION = {
    'max_frames': 64,
    'input_size': 63,
    'num_classes': NUM_CLASSES,
    
    # Arquitectura
    'hidden_size': 256,
    'num_layers': 2,
    'dropout': 0.4,
    'bidirectional': True,
    
    # Training
    'batch_size': 32,
    'epochs': 150,
    'lr': 1e-3,
    'weight_decay': 5e-4,
    'label_smoothing': 0.1,
    'patience': 30,
}

print("Config GRU+Attention:")
for k, v in CONFIG_ATTENTION.items():
    print(f"  {k}: {v}")

Config GRU+Attention:
  max_frames: 64
  input_size: 63
  num_classes: 30
  hidden_size: 256
  num_layers: 2
  dropout: 0.4
  bidirectional: True
  batch_size: 32
  epochs: 150
  lr: 0.001
  weight_decay: 0.0005
  label_smoothing: 0.1
  patience: 30


In [16]:
# ==========================================
# 16. CREAR DATASETS PARA ENTRENAMIENTO
# ==========================================

# Usar dataset enriquecido si existe
TRAIN_ROOT = FILTERED_ROOT if FILTERED_ROOT.exists() else WLASL30_ROOT

train_aug = SkeletalAugmentation()
train_dataset_attn = PoseDatasetTTA(TRAIN_ROOT, 'train', gloss_to_idx, augmentation=train_aug)
val_dataset_attn = PoseDatasetTTA(TRAIN_ROOT, 'val', gloss_to_idx)

train_loader_attn = DataLoader(train_dataset_attn, batch_size=32, shuffle=True, num_workers=0)
val_loader_attn = DataLoader(val_dataset_attn, batch_size=32, shuffle=False)

print(f"Train: {len(train_dataset_attn)} samples ({TRAIN_ROOT.name})")
print(f"Val: {len(val_dataset_attn)} samples")

Train: 242 samples (wlasl100_filtered30)
Val: 56 samples


In [17]:
# ==========================================
# 17. ENTRENAR GRU+ATTENTION
# ==========================================

model_attention = PoseClassifierAttention(CONFIG_ATTENTION).to(device)
print(f"Parametros: {sum(p.numel() for p in model_attention.parameters()):,}")

criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG_ATTENTION['label_smoothing'])
optimizer = AdamW(model_attention.parameters(), lr=CONFIG_ATTENTION['lr'],
                  weight_decay=CONFIG_ATTENTION['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG_ATTENTION['epochs'])

history_attn = {'train_acc': [], 'val_top1': [], 'val_top5': []}
best_val = 0
patience_counter = 0

print("\n" + "=" * 60)
print("ENTRENAMIENTO: GRU + ATTENTION")
print("=" * 60)

for epoch in range(CONFIG_ATTENTION['epochs']):
    # Train
    model_attention.train()
    correct, total = 0, 0
    
    for batch, labels in train_loader_attn:
        batch, labels = batch.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_attention(batch)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_attention.parameters(), 1.0)
        optimizer.step()
        
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100 * correct / total
    scheduler.step()
    
    # Eval
    val_metrics = evaluate_standard([model_attention], val_loader_attn)
    
    history_attn['train_acc'].append(train_acc)
    history_attn['val_top1'].append(val_metrics['top1_acc'])
    history_attn['val_top5'].append(val_metrics['top5_acc'])
    
    if val_metrics['top1_acc'] > best_val:
        best_val = val_metrics['top1_acc']
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_attention.state_dict(),
            'val_top1': best_val,
            'config': CONFIG_ATTENTION
        }, OUTPUT_DIR / 'best_model_attention.pt')
    else:
        patience_counter += 1
    
    if patience_counter >= CONFIG_ATTENTION['patience']:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_acc:.1f}% | Val: {val_metrics['top1_acc']:.1f}% (Top-5: {val_metrics['top5_acc']:.1f}%) | Best: {best_val:.1f}%")

print(f"\nMejor Val Top-1: {best_val:.1f}% (epoch {best_epoch})")

Parametros: 2,259,742

ENTRENAMIENTO: GRU + ATTENTION
Epoch   1 | Train: 2.5% | Val: 3.6% (Top-5: 19.6%) | Best: 3.6%
Epoch  10 | Train: 44.6% | Val: 41.1% (Top-5: 76.8%) | Best: 46.4%
Epoch  20 | Train: 73.1% | Val: 53.6% (Top-5: 83.9%) | Best: 55.4%
Epoch  30 | Train: 89.7% | Val: 62.5% (Top-5: 92.9%) | Best: 62.5%
Epoch  40 | Train: 96.7% | Val: 64.3% (Top-5: 85.7%) | Best: 64.3%
Epoch  50 | Train: 97.1% | Val: 57.1% (Top-5: 82.1%) | Best: 64.3%
Epoch  60 | Train: 100.0% | Val: 58.9% (Top-5: 83.9%) | Best: 64.3%

Early stopping at epoch 65

Mejor Val Top-1: 64.3% (epoch 35)


In [18]:
# ==========================================
# 18. CARGAR MEJOR MODELO ATTENTION
# ==========================================

ckpt = torch.load(OUTPUT_DIR / 'best_model_attention.pt')
model_attention.load_state_dict(ckpt['model_state_dict'])
model_attention.eval()

# Evaluar en test original
test_attn = evaluate_standard([model_attention], test_loader)
print(f"GRU+Attention Test: Top-1: {test_attn['top1_acc']:.1f}%  Top-5: {test_attn['top5_acc']:.1f}%")

GRU+Attention Test: Top-1: 58.8%  Top-5: 91.2%


---
# PARTE 3: ENTRENAR TRANSFORMER (Opcional)
---

In [19]:
# ==========================================
# 19. CONFIGURACION TRANSFORMER
# ==========================================

CONFIG_TRANSFORMER = {
    'max_frames': 64,
    'input_size': 63,
    'num_classes': NUM_CLASSES,
    
    # Arquitectura Transformer
    'd_model': 256,
    'nhead': 8,
    'num_transformer_layers': 4,
    'dropout': 0.3,  # Menos dropout para Transformer
    
    # Training
    'batch_size': 32,
    'epochs': 150,
    'lr': 5e-4,  # LR mas bajo para Transformer
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'patience': 30,
}

print("Config Transformer:")
print(f"  d_model: {CONFIG_TRANSFORMER['d_model']}")
print(f"  nhead: {CONFIG_TRANSFORMER['nhead']}")
print(f"  layers: {CONFIG_TRANSFORMER['num_transformer_layers']}")

Config Transformer:
  d_model: 256
  nhead: 8
  layers: 4


In [20]:
# ==========================================
# 20. ENTRENAR TRANSFORMER
# ==========================================

model_transformer = PoseTransformer(CONFIG_TRANSFORMER).to(device)
print(f"Parametros Transformer: {sum(p.numel() for p in model_transformer.parameters()):,}")

criterion_t = nn.CrossEntropyLoss(label_smoothing=CONFIG_TRANSFORMER['label_smoothing'])
optimizer_t = AdamW(model_transformer.parameters(), lr=CONFIG_TRANSFORMER['lr'],
                    weight_decay=CONFIG_TRANSFORMER['weight_decay'])
scheduler_t = CosineAnnealingLR(optimizer_t, T_max=CONFIG_TRANSFORMER['epochs'])

history_transformer = {'train_acc': [], 'val_top1': [], 'val_top5': []}
best_val_t = 0
patience_counter_t = 0

print("\n" + "=" * 60)
print("ENTRENAMIENTO: TRANSFORMER")
print("=" * 60)

for epoch in range(CONFIG_TRANSFORMER['epochs']):
    model_transformer.train()
    correct, total = 0, 0
    
    for batch, labels in train_loader_attn:
        batch, labels = batch.to(device), labels.to(device)
        
        optimizer_t.zero_grad()
        outputs = model_transformer(batch)
        loss = criterion_t(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_transformer.parameters(), 1.0)
        optimizer_t.step()
        
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100 * correct / total
    scheduler_t.step()
    
    val_metrics = evaluate_standard([model_transformer], val_loader_attn)
    
    history_transformer['train_acc'].append(train_acc)
    history_transformer['val_top1'].append(val_metrics['top1_acc'])
    history_transformer['val_top5'].append(val_metrics['top5_acc'])
    
    if val_metrics['top1_acc'] > best_val_t:
        best_val_t = val_metrics['top1_acc']
        best_epoch_t = epoch + 1
        patience_counter_t = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_transformer.state_dict(),
            'val_top1': best_val_t,
            'config': CONFIG_TRANSFORMER
        }, OUTPUT_DIR / 'best_model_transformer.pt')
    else:
        patience_counter_t += 1
    
    if patience_counter_t >= CONFIG_TRANSFORMER['patience']:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_acc:.1f}% | Val: {val_metrics['top1_acc']:.1f}% (Top-5: {val_metrics['top5_acc']:.1f}%) | Best: {best_val_t:.1f}%")

print(f"\nMejor Val Top-1: {best_val_t:.1f}% (epoch {best_epoch_t})")

Parametros Transformer: 3,213,470

ENTRENAMIENTO: TRANSFORMER
Epoch   1 | Train: 3.7% | Val: 3.6% (Top-5: 21.4%) | Best: 3.6%
Epoch  10 | Train: 24.8% | Val: 23.2% (Top-5: 53.6%) | Best: 23.2%
Epoch  20 | Train: 49.2% | Val: 23.2% (Top-5: 69.6%) | Best: 28.6%
Epoch  30 | Train: 67.4% | Val: 33.9% (Top-5: 80.4%) | Best: 44.6%
Epoch  40 | Train: 78.9% | Val: 39.3% (Top-5: 80.4%) | Best: 44.6%
Epoch  50 | Train: 89.3% | Val: 44.6% (Top-5: 75.0%) | Best: 46.4%
Epoch  60 | Train: 93.4% | Val: 46.4% (Top-5: 75.0%) | Best: 50.0%
Epoch  70 | Train: 96.3% | Val: 46.4% (Top-5: 75.0%) | Best: 50.0%
Epoch  80 | Train: 99.6% | Val: 41.1% (Top-5: 73.2%) | Best: 50.0%

Early stopping at epoch 87

Mejor Val Top-1: 50.0% (epoch 57)


In [21]:
# ==========================================
# 21. CARGAR MEJOR TRANSFORMER
# ==========================================

ckpt_t = torch.load(OUTPUT_DIR / 'best_model_transformer.pt')
model_transformer.load_state_dict(ckpt_t['model_state_dict'])
model_transformer.eval()

test_transformer = evaluate_standard([model_transformer], test_loader)
print(f"Transformer Test: Top-1: {test_transformer['top1_acc']:.1f}%  Top-5: {test_transformer['top5_acc']:.1f}%")

Transformer Test: Top-1: 50.0%  Top-5: 73.5%


---
# PARTE 4: ENSEMBLE FINAL CON TTA
---

In [22]:
# ==========================================
# 22. ENSEMBLE FINAL
# ==========================================

print("\n" + "=" * 70)
print("ENSEMBLE FINAL: TODOS LOS MODELOS")
print("=" * 70)

# Recopilar todos los modelos
all_models = list(models_loaded.values())
all_names = list(models_loaded.keys())

# Añadir Attention si existe
if (OUTPUT_DIR / 'best_model_attention.pt').exists():
    all_models.append(model_attention)
    all_names.append('GRU_Attention')

# Añadir Transformer si existe
if (OUTPUT_DIR / 'best_model_transformer.pt').exists():
    all_models.append(model_transformer)
    all_names.append('Transformer')

print(f"\nModelos en ensemble: {all_names}")


ENSEMBLE FINAL: TODOS LOS MODELOS

Modelos en ensemble: ['LSTM', 'GRU', 'GRU_Enriched', 'GRU_Attention', 'Transformer']


In [23]:
# ==========================================
# 23. EVALUACION FINAL
# ==========================================

print("\n--- Resultados Individuales (sin TTA) ---")
for name, model in zip(all_names, all_models):
    model.eval()
    val_m = evaluate_standard([model], val_loader)
    test_m = evaluate_standard([model], test_loader)
    print(f"{name:<20} Val: {val_m['top1_acc']:.1f}%  Test: {test_m['top1_acc']:.1f}% (Top-5: {test_m['top5_acc']:.1f}%)")


--- Resultados Individuales (sin TTA) ---
LSTM                 Val: 37.5%  Test: 41.2% (Top-5: 67.6%)
GRU                  Val: 53.6%  Test: 38.2% (Top-5: 76.5%)
GRU_Enriched         Val: 51.8%  Test: 47.1% (Top-5: 79.4%)
GRU_Attention        Val: 64.3%  Test: 58.8% (Top-5: 91.2%)
Transformer          Val: 50.0%  Test: 50.0% (Top-5: 73.5%)


In [24]:
# ==========================================
# 24. ENSEMBLE SIN TTA
# ==========================================

print("\n--- Ensemble (sin TTA) ---")
ensemble_val = evaluate_standard(all_models, val_loader)
ensemble_test = evaluate_standard(all_models, test_loader)
print(f"Ensemble ({len(all_models)} modelos): Val: {ensemble_val['top1_acc']:.1f}%  Test: {ensemble_test['top1_acc']:.1f}% (Top-5: {ensemble_test['top5_acc']:.1f}%)")


--- Ensemble (sin TTA) ---
Ensemble (5 modelos): Val: 57.1%  Test: 58.8% (Top-5: 79.4%)


In [25]:
# ==========================================
# 25. ENSEMBLE CON TTA (LA BOMBA FINAL)
# ==========================================

print("\n--- Ensemble + TTA (5 augmentaciones) ---")
ensemble_val_tta = evaluate_with_tta(all_models, val_dataset, n_augments=5)
ensemble_test_tta = evaluate_with_tta(all_models, test_dataset, n_augments=5)
print(f"Ensemble+TTA: Val: {ensemble_val_tta['top1_acc']:.1f}%  Test: {ensemble_test_tta['top1_acc']:.1f}% (Top-5: {ensemble_test_tta['top5_acc']:.1f}%)")


--- Ensemble + TTA (5 augmentaciones) ---


TTA Evaluation: 100%|██████████| 34/34 [00:00<00:00, 58.88it/s]

Ensemble+TTA: Val: 55.4%  Test: 58.8% (Top-5: 79.4%)


In [26]:
# ==========================================
# 26. RESUMEN FINAL
# ==========================================

print("\n" + "=" * 80)
print("RESUMEN FINAL - CAMINO AL 60%")
print("=" * 80)
print("")
print(f"{'Configuracion':<40} {'Val Top-1':<12} {'Test Top-1':<12} {'Test Top-5':<12}")
print("-" * 80)

# Individuales
for name, model in zip(all_names, all_models):
    val_m = evaluate_standard([model], val_loader)
    test_m = evaluate_standard([model], test_loader)
    print(f"{name:<40} {val_m['top1_acc']:.1f}%{'':<7} {test_m['top1_acc']:.1f}%{'':<7} {test_m['top5_acc']:.1f}%")

print("-" * 80)

# Ensembles
print(f"{'Ensemble (sin TTA)':<40} {ensemble_val['top1_acc']:.1f}%{'':<7} {ensemble_test['top1_acc']:.1f}%{'':<7} {ensemble_test['top5_acc']:.1f}%")
print(f"{'Ensemble + TTA':<40} {ensemble_val_tta['top1_acc']:.1f}%{'':<7} {ensemble_test_tta['top1_acc']:.1f}%{'':<7} {ensemble_test_tta['top5_acc']:.1f}%")

print("")
print("=" * 80)
print(f"OBJETIVO DEL TUTOR: Top-1 60-70%")

best_test = ensemble_test_tta['top1_acc']
if best_test >= 60:
    print(f"\n*** OBJETIVO ALCANZADO: {best_test:.1f}% ***")
else:
    print(f"\nMejor resultado: {best_test:.1f}%")
    print(f"Faltan: {60 - best_test:.1f} puntos para el objetivo")


RESUMEN FINAL - CAMINO AL 60%

Configuracion                            Val Top-1    Test Top-1   Test Top-5  
--------------------------------------------------------------------------------
LSTM                                     37.5%        41.2%        67.6%
GRU                                      53.6%        38.2%        76.5%
GRU_Enriched                             51.8%        47.1%        79.4%
GRU_Attention                            64.3%        58.8%        91.2%
Transformer                              50.0%        50.0%        73.5%
--------------------------------------------------------------------------------
Ensemble (sin TTA)                       57.1%        58.8%        79.4%
Ensemble + TTA                           55.4%        58.8%        79.4%

OBJETIVO DEL TUTOR: Top-1 60-70%

Mejor resultado: 58.8%
Faltan: 1.2 puntos para el objetivo


In [27]:
# ==========================================
# 27. GUARDAR RESULTADOS
# ==========================================

results = {
    'models': all_names,
    'individual_results': {},
    'ensemble_no_tta': {'val': ensemble_val, 'test': ensemble_test},
    'ensemble_with_tta': {'val': ensemble_val_tta, 'test': ensemble_test_tta},
    'config_attention': CONFIG_ATTENTION,
    'config_transformer': CONFIG_TRANSFORMER,
}

for name, model in zip(all_names, all_models):
    val_m = evaluate_standard([model], val_loader)
    test_m = evaluate_standard([model], test_loader)
    results['individual_results'][name] = {'val': val_m, 'test': test_m}

with open(OUTPUT_DIR / 'final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResultados guardados en: {OUTPUT_DIR}")


Resultados guardados en: /home/fransando/Dev/tfg-sign-language-recognition/05_experiments/final_assault


## Conclusiones

### Si llegaste al 60%:
Enhorabuena! El ensemble de multiples arquitecturas + TTA es muy robusto.

### Si no llegaste:
Opciones adicionales:
1. **Mas TTA**: Subir a 10-15 augmentaciones
2. **Weighted ensemble**: Dar mas peso a los mejores modelos
3. **Mixup/Cutmix**: Data augmentation a nivel de batch
4. **Mas datos**: Usar WLASL completo (2000 clases) y filtrar top 30 con mas videos
5. **Knowledge distillation**: Entrenar modelo estudiante del ensemble